**Startup:** run Steps 1–5, then 7–9. Run teardown only after the local episode finishes.

# Colab PolicyServer + Tunnel Launch Instructions

Phase 11, Plan 11-03, Task 3. Documents how to stand up the Colab-hosted
`lerobot.async_inference.PolicyServer` and connect this project's local
`RobotClient` to it over a tunnel, before Plan 11-04 writes any code that
assumes this bridge exists.

**Selected checkpoint (Task 2's `checkpoint:decision`):** `victorvanhalst/smolvla_so101_cube`
— chosen for its exact task-instruction match to D-02's red-cube task. See
`checkpoint_candidates.md` for the fetched `config.json` data and
`11-03-SUMMARY.md` for the full decision rationale.

**pyngrok legitimacy gate:** this file's step 2 installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

---

## Camera mapping (Task 2's split-stereo resolution)

`victorvanhalst/smolvla_so101_cube`'s `config.json` expects 3 named camera
inputs (`camera1`, `camera2`, `camera3` — none marked `empty_camera_*`).
This project's real rig has only 2 physical cameras:

- IMX335 wrist camera, cv2 index 0, native 1920x1080
- AR0144 stereo camera, cv2 index 1, captured at 1600x600 (800x600 per
  half) — **one physical side-by-side stereo frame, not two independent
  feeds** (confirmed in `diagnostics/UAT/function/basic/UAT.md`, Step 4).
  1600x600 was chosen over the previously-implemented 1280x360 for
  meaningfully higher per-eye resolution while remaining confirmed stable;
  the still-broken 2560x720 mode remains frozen even after exhausting the
  last known macOS system-level fix — see `stereo_camera.py`'s module
  docstring for the full diagnosis.

Resolution: split the AR0144's single 1600x600 frame into left/right
halves, giving **3 genuinely distinct real camera views** (not a dummy or
duplicated 3rd slot):

| Checkpoint camera key | Real source                 | Crop                                         |
| --------------------- | --------------------------- | -------------------------------------------- |
| `camera1`           | IMX335 wrist                | as-is, full 1920x1080 frame                  |
| `camera2`           | AR0144 stereo — LEFT half  | columns`[0:800]` of the 1600x600 frame    |
| `camera3`           | AR0144 stereo — RIGHT half | columns`[800:1600]` of the 1600x600 frame |

This is a better structural fit than an empty/dummy 3rd slot: all 3 of the
checkpoint's expected inputs receive real image data during inference,
none is a zero/black placeholder. (Community precedent: other SmolVLA
fine-tunes such as `bklassen3434/smolvla_pick_pen_v2_frozen` and
`Yilin1001/smolvla-pen-v2-030000` pair a 2-real-camera rig with an
inherited 3rd config slot by leaving it a dummy — this project instead has
a 3rd genuinely distinct real view available via the stereo split, which
is a strictly better fit than a dummy slot would be.)

`lerobot.async_inference.robot_client`'s `--robot.cameras` flag takes a
dict of named camera configs, each independently opened by `cv2` — it has
no built-in "crop a region of a wider frame" option. Two ways to produce
`camera2`/`camera3` as independent feeds from the one AR0144 device:

1. **Preferred for this bridge:** a thin wrapper camera process/script that
   opens cv2 index 1 once, splits each frame into left/right halves, and
   republishes them as two separate virtual camera endpoints the
   `RobotClient`'s camera config can point at (e.g. two loopback
   `cv2.VideoCapture`-compatible sources, or if `lerobot`'s camera config
   supports a custom `type` plugin, register a "split-half" camera type
   that does the crop inline). This avoids opening the same physical
   device twice (most webcam drivers do not support concurrent opens of
   one index).
2. **Fallback if (1) isn't scriptable in time:** open cv2 index 1 once in
   the `RobotClient`'s local process, capture the full 1600x600 frame per
   step, crop it into `camera2`/`camera3` numpy arrays in the observation
   dict before it's sent to the `PolicyServer` — this requires a small
   patch/subclass of the stock `robot_client.py` observation-building step
   rather than pure CLI flags, since the stock CLI only supports
   independent `cv2` device opens per named camera.

Plan 11-04 (which writes `robot_client.py`-adjacent bridge code) is
responsible for implementing whichever of these two options it lands on;
this file only fixes the *mapping* (which checkpoint key gets which real
data), not the implementation mechanism.

---

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

OpenMP duplicate-runtime guards set.


## Step 1 — Colab: install `lerobot[async]`

In [2]:
!pip install 'lerobot[async]==0.6.1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 54.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 197.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 49.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.

Adds `grpcio` — already audited and approved in `11-RESEARCH.md`'s Package
Legitimacy Audit as a required transitive dependency of `lerobot`'s own
`async` extra. No separate checkpoint needed for this install.

## Step 2 — Colab: install `pyngrok` (gated — see legitimacy note above)

In [3]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

## Step 3 — Colab: start the `PolicyServer`

In [4]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 22.2 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=8f997e80515dd2609cb3c14fba73389b3b525bbc08171ae1e0366795c4a49ea6
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [5]:
!pip install 'lerobot[dataset]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 65.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 85.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [6]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

Drive mount/token read skipped (MessageError: User cancelled dfs_ephemeral authorization)
Logged into HF Hub.
ngrok authtoken set.                                                                                


`lerobot`'s installs (Step 1's `lerobot[async]` and the earlier `lerobot[dataset]` cell) pull in a `torchaudio` build whose CUDA version doesn't always match Colab's preinstalled `torch` -- confirmed live (Phase 12 Test 7 UAT) to crash Step 3's `from lerobot.async_inference.policy_server import serve` with `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions`, which cascades into a `ModuleNotFoundError` for `transformers.AutoProcessor` and can leave the kernel needing a full restart. `torchaudio` isn't actually used by anything in this notebook -- uninstalling it removes the mismatched import path entirely (`transformers`' own `is_torchaudio_available()` check then correctly skips it).

In [7]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [8]:
import os
import socket
import subprocess
import sys
import time

POLICY_SERVER_LOG = "/content/policy_server.log"

# Test 7 UAT (Phase 12, 2026-09-27) root-cause work: `serve()` running as a
# Python threading.Thread inside THIS kernel process meant a process-level
# crash (OOM-kill, CUDA driver segfault) took the whole kernel down with it
# -- Flask depth server, both tunnels, everything -- leaving only "The Kernel
# crashed while executing code in the current cell or a previous cell", not a
# catchable Python exception. The saved cell output also showed the crash
# happened mid-load of SmolVLA's VLM backbone, in the SAME kernel process
# where Step 7 later does `pip install torch==2.6.0 ... --index-url
# .../cu124` -- a version/CUDA-build reinstall happening AFTER this cell had
# already imported the original torch into memory.
#
# Launching PolicyServer as a genuine, separate OS subprocess instead means:
#   - a crash in it cannot take this kernel (or the depth server/tunnels) down
#   - its stdout/stderr go straight to a file via the OS, independent of
#     Jupyter's per-cell output capture, so the log survives even a hard crash
#   - it re-imports torch fresh from disk at subprocess start, so it can
#     never end up running on a stale in-process torch that predates a later
#     `pip install torch==...` -- each subprocess only ever sees whatever is
#     on disk when IT starts, fully isolated from Step 7's reinstall
#   - CUDA_LAUNCH_BLOCKING=1 forces synchronous CUDA execution, so an async
#     driver-level crash surfaces as a normal Python traceback in the log
#     instead of an opaque process death
#   - KMP_DUPLICATE_LIB_OK / OMP_NUM_THREADS (also set globally above) guard
#     this subprocess too against the "OMP: Error #15" duplicate-runtime crash

# Kill any stray PolicyServer/run_demo.py processes left over from a
# previous attempt tonight -- a hung child from an earlier crash keeps
# holding GPU memory, which can itself cause the NEXT model load to OOM.
subprocess.run(["pkill", "-9", "-f", "lerobot.async_inference.policy_server"], check=False)
time.sleep(1)

print("--- GPU memory before starting PolicyServer ---")
subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"],
    check=False,
)

_env = os.environ.copy()
_env["CUDA_LAUNCH_BLOCKING"] = "1"
_env.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
_env.setdefault("OMP_NUM_THREADS", "1")

_log_fh = open(POLICY_SERVER_LOG, "w", buffering=1)
policy_server_process = subprocess.Popen(
    [
        sys.executable, "-m", "lerobot.async_inference.policy_server",
        "--host", "0.0.0.0", "--port", "5173",
    ],
    stdout=_log_fh,
    stderr=subprocess.STDOUT,
    env=_env,
)

# Wait for the server to actually be listening before Step 4 opens the
# tunnel against it (subprocess startup is not instant).
for _ in range(60):
    if policy_server_process.poll() is not None:
        raise RuntimeError(
            f"PolicyServer subprocess exited early (code {policy_server_process.returncode}) "
            f"-- see {POLICY_SERVER_LOG}"
        )
    try:
        with socket.create_connection(("127.0.0.1", 5173), timeout=1):
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(f"PolicyServer did not start listening in time -- see {POLICY_SERVER_LOG}")

print(f"PolicyServer running as PID {policy_server_process.pid}. Logs -> {POLICY_SERVER_LOG}")
print(f"Tail it live from another cell with: !tail -f {POLICY_SERVER_LOG}")
print("Check it's still alive from another cell with: policy_server_process.poll()  # None = still running")

--- GPU memory before starting PolicyServer ---
PolicyServer running as PID 1080. Logs -> /content/policy_server.log
Tail it live from another cell with: !tail -f /content/policy_server.log
Check it's still alive from another cell with: policy_server_process.poll()  # None = still running


PolicyServer now runs as a real OS subprocess (`policy_server_process`), not
a background thread -- Colab's async-cell execution note above no longer
applies, but the same idea holds: this cell returns immediately once the
server is confirmed listening, so the notebook can go straight on to the
tunnel cell below while the subprocess keeps running independently.

## Step 4 — Colab: open a TCP tunnel (not HTTP)

In [9]:
# Colab notebook cell
from pyngrok import ngrok

# Explicitly request a TCP tunnel — gRPC needs HTTP/2 semantics that a
# default HTTP tunnel silently breaks (Pitfall 5 in 11-RESEARCH.md).
tunnel = ngrok.connect(5173, "tcp")
print(f"PolicyServer reachable at: {tunnel.public_url}")
# public_url looks like tcp://0.tcp.ngrok.io:12345 — strip the "tcp://"
# prefix; the local RobotClient's --server_address flag wants host:port.

PolicyServer reachable at: tcp://4.tcp.ngrok.io:17093


In [10]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True NVIDIA L4


In [16]:
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total --format=csv

utilization.gpu [%], utilization.memory [%], memory.used [MiB], memory.total [MiB]
0 %, 0 %, 2802 MiB, 23034 MiB


Copy the printed `host:port` (after stripping `tcp://`) for Step 5.

If ngrok's free tier no longer supports a TCP tunnel endpoint by the time
this is run (Assumption A4 in `11-RESEARCH.md`), fall back to `cloudflared`
with a configured Zero Trust tunnel for raw TCP forwarding (heavier setup
than ngrok TCP — cloudflared's zero-config "quick tunnels" are HTTP-only
and will not work here).

## Step 5 — Local (`control/`): launch the `RobotClient`

Notes:

- `<ngrok_tcp_host>:<ngrok_tcp_port>` = the value printed in Step 4, with
  the `tcp://` prefix stripped.
- `--robot.port` / `--robot.id` match `control/COMMANDS.md`'s documented
  follower arm (`/dev/cu.usbmodem5B8E1139151`, `soarm_follower_02`) — reverify
  via `ls /dev/cu.usbmodem*` if the port has been reassigned since last use.
- The `camera2`/`camera3` `type: <split-stereo-left/right>` placeholders are
  **not** stock `lerobot` camera types — see the "Camera mapping" section
  above; Plan 11-04 implements the actual split mechanism (custom camera
  type plugin, or a `robot_client.py` observation-patch). This file
  documents the target mapping, not that implementation.
- `--task` uses the exact instruction wording from the checkpoint's own
  training task ("Pick the red cube and place it in the bowl"), not a
  paraphrase — the model card / task description string should be
  reverified against the actual checkpoint metadata before the live run in
  Plan 11-05, since exact wording can matter for language-conditioned
  policies.
- `--policy_device=cuda` assumes the `PolicyServer` runs on a Colab GPU
  runtime; the policy device is a Colab-side concern (loaded by
  `PolicyServerConfig`/`serve()` in Step 3), this flag on the local CLI
  side only affects what the `RobotClient` requests be reported.

The cell above now terminates the `PolicyServer` subprocess directly
(`policy_server_process.terminate()`); no need to click a cell's stop button
or restart the runtime for it, since it's no longer running inside the
notebook kernel itself (2026-09-27 subprocess-isolation fix).

---

### Step 7 — Colab: install and load Fast-FoundationStereo

In [10]:
import subprocess
from pathlib import Path
FASTFS_DIR = Path('/content/Fast-FoundationStereo')
if not (FASTFS_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/NVlabs/Fast-FoundationStereo', str(FASTFS_DIR)], check=True)
print('FastFS source revision:', subprocess.check_output(['git', '-C', str(FASTFS_DIR), 'rev-parse', 'HEAD'], text=True).strip())

FastFS source revision: 476f4249561f7c79ca707326954f9255643412a6


In [11]:
# Separate filesystem environment: never reinstall torch/cuDNN under a running PolicyServer.
import sys
import subprocess
from pathlib import Path
FASTFS_VENV = '/content/fastfs_venv'
FASTFS_PYTHON = f'{FASTFS_VENV}/bin/python'
if 'depth_server_process' in globals() and depth_server_process.poll() is None:
    raise RuntimeError('Stop the depth service before changing its environment; use teardown below.')
if not Path(FASTFS_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', FASTFS_VENV], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0', 'xformers',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', '-r', '/content/Fast-FoundationStereo/requirements.txt'], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', 'flask'], check=True)
# No GUI patch: the resident adapter calls the model directly, never run_demo.py.

CompletedProcess(args=['/content/fastfs_venv/bin/python', '-m', 'pip', 'install', 'flask'], returncode=0)

Use the trusted NVlabs `23-36-37` research checkpoint below. Step 8 loads it once
in the isolated environment, warms inference, then serves subsequent requests in
process. No CLI subprocess or model reload occurs per depth request.

Download the checkpoint and configuration into the isolated service’s weights directory:

In [12]:
import os

os.makedirs("/content/Fast-FoundationStereo/weights/23-36-37", exist_ok=True)

!pip install -q gdown
!gdown "1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth
!gdown "1GDBRYL-ZaLpXEtWfGFRJvkBc_2sywjgj" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/cfg.yaml

print(os.listdir("/content/Fast-FoundationStereo/weights/23-36-37"))

Downloading...
From (original): https://drive.google.com/uc?id=1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E
From (redirected): https://drive.google.com/uc?id=1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E&confirm=t&uuid=d121ca92-5770-4c4c-ad14-099027af2737
To: /content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth
100% 71.1M/71.1M [00:03<00:00, 19.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GDBRYL-ZaLpXEtWfGFRJvkBc_2sywjgj
To: /content/Fast-FoundationStereo/weights/23-36-37/cfg.yaml
100% 182/182 [00:00<00:00, 1.02MB/s]
['model_best_bp2_serialize.pth', 'cfg.yaml']


### Step 8 — Start the resident depth service
Run both cells below. The first writes the versioned adapter (embedded so this
notebook is standalone). The second launches it as a separate OS process using
FastFS’s virtualenv and waits for warmup. Keep the model resident for the entire
episode. `/health` only becomes available after warmup. Logs survive notebook errors
at `/content/fastfs_server.log`.

In [13]:
%%writefile /content/fastfs_server.py
"""Resident FastFS HTTP service. Run with the isolated FastFS Python interpreter.

Adapter follows NVlabs/Fast-FoundationStereo scripts/run_demo.py's model API;
no demo subprocess, GUI, point-cloud construction, or intermediate files.
"""
import argparse
import base64
import hashlib
import io
import threading
import time
from pathlib import Path

import cv2
import numpy as np


class FastFSModel:
    def __init__(self, checkpoint, valid_iters=8, max_disp=192):
        import torch
        from core.utils.utils import InputPadder
        from Utils import AMP_DTYPE

        self.torch, self.padder, self.amp_dtype = torch, InputPadder, AMP_DTYPE
        if not torch.cuda.is_available():
            raise RuntimeError("FastFS requires CUDA in its isolated environment")
        checkpoint = Path(checkpoint).resolve()
        digest = hashlib.sha256()
        with checkpoint.open('rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''):
                digest.update(block)
        # Research checkpoints serialize the complete model. Only load the
        # operator-selected, trusted NVlabs checkpoint, never request data.
        self.model = torch.load(checkpoint, map_location='cpu', weights_only=False)
        self.model.args.valid_iters = valid_iters
        self.model.args.max_disp = max_disp
        self.model.cuda().eval()
        self.valid_iters = valid_iters
        self.metadata = dict(checkpoint=checkpoint.name, checkpoint_sha256=digest.hexdigest(),
                             valid_iters=valid_iters, max_disp=max_disp, units='metres')

    def __call__(self, left_bgr, right_bgr, intrinsics, baseline):
        torch = self.torch
        tensors = [torch.from_numpy(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)).to('cuda', dtype=torch.float32)
                   .permute(2, 0, 1).unsqueeze(0) for x in (left_bgr, right_bgr)]
        padding = self.padder(tensors[0].shape, divis_by=32, force_square=False)
        left, right = padding.pad(*tensors)
        with torch.inference_mode(), torch.amp.autocast('cuda', dtype=self.amp_dtype):
            disparity = self.model.forward(left, right, iters=self.valid_iters,
                                           test_mode=True, optimize_build_volume='pytorch1')
        disparity = padding.unpad(disparity.float()).cpu().numpy().reshape(left_bgr.shape[:2])
        depth = np.full(disparity.shape, np.nan, dtype=np.float32)
        valid = np.isfinite(disparity) & (disparity > 0)
        np.divide(float(intrinsics[0]) * baseline, disparity, out=depth, where=valid)
        return depth


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/depth')
    def depth():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='depth service busy'), 429
        try:
            body = request.get_json()
            frames = []
            for key in ('left_png_b64', 'right_png_b64'):
                raw = base64.b64decode(body[key], validate=True)
                frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
                if frame is None:
                    raise ValueError('invalid PNG')
                frames.append(frame)
            if frames[0].shape != frames[1].shape or max(frames[0].shape[:2]) > 992:
                raise ValueError('expected equal stereo images with dimensions <=992')
            K = np.asarray(body['intrinsics_flat'], dtype=float)
            baseline = float(body['baseline_m'])
            if K.shape != (9,) or not np.isfinite(K).all() or K[0] <= 0 or not np.isfinite(baseline) or baseline <= 0:
                raise ValueError('invalid calibration')
            started = time.perf_counter()
            result = infer(*frames, K.tolist(), baseline)
            elapsed = (time.perf_counter() - started) * 1000
            buf = io.BytesIO()
            np.save(buf, np.asarray(result, dtype=np.float32), allow_pickle=False)
            return jsonify(depth_npy_b64=base64.b64encode(buf.getvalue()).decode('ascii'),
                           model=infer.metadata, inference_ms=elapsed,
                           request_id=body.get('request_id'))
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        finally:
            inference_lock.release()
    return app


def main():
    import sys
    parser = argparse.ArgumentParser()
    parser.add_argument('--repo', required=True)
    parser.add_argument('--checkpoint', required=True)
    parser.add_argument('--port', type=int, default=3000)
    parser.add_argument('--warmup-width', type=int, default=800)
    parser.add_argument('--warmup-height', type=int, default=576)
    args = parser.parse_args()
    sys.path.insert(0, str(Path(args.repo).resolve()))
    model = FastFSModel(args.checkpoint)
    # Compilation happens before the socket opens and readiness is reported.
    warmup = np.zeros((args.warmup_height, args.warmup_width, 3), dtype=np.uint8)
    model(warmup, warmup, [800., 0., 400., 0., 800., 288., 0., 0., 1.], .05)
    print('FastFS model loaded and warmed; ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


Writing /content/fastfs_server.py


In [14]:
import os
import json
import time
import urllib.request
import subprocess
from pathlib import Path
FASTFS_PYTHON = '/content/fastfs_venv/bin/python'
FASTFS_DIR = '/content/Fast-FoundationStereo'
FASTFS_MODEL = f'{FASTFS_DIR}/weights/23-36-37/model_best_bp2_serialize.pth'
DEPTH_SERVER_LOG = '/content/fastfs_server.log'
if not Path(FASTFS_MODEL).is_file():
    raise FileNotFoundError(f'Checkpoint missing: {FASTFS_MODEL}; complete Step 7 first')
if 'depth_server_process' in globals() and depth_server_process.poll() is None:
    depth_server_process.terminate()
    try:
        depth_server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        depth_server_process.kill()
        depth_server_process.wait()
with __import__('socket').socket() as probe:
    if probe.connect_ex(('127.0.0.1', 3000)) == 0:
        raise RuntimeError('Port 3000 is already occupied. Stop the old depth service or start a fresh runtime.')
env = os.environ.copy()
env['PYTHONPATH'] = FASTFS_DIR
env.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
env.setdefault('OMP_NUM_THREADS', '1')
with open(DEPTH_SERVER_LOG, 'w') as log:
    depth_server_process = subprocess.Popen(
        [FASTFS_PYTHON, '-u', '/content/fastfs_server.py', '--repo', FASTFS_DIR,
         '--checkpoint', FASTFS_MODEL, '--port', '3000'],
        cwd=FASTFS_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
print('Loading and warming FastFS once. First compilation may take several minutes.')
deadline = time.monotonic() + 600
while time.monotonic() < deadline:
    if depth_server_process.poll() is not None:
        raise RuntimeError(Path(DEPTH_SERVER_LOG).read_text()[-6000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:3000/health', timeout=2) as response:
            health = json.load(response)
        if health.get('status') == 'ready' and health.get('protocol_version') == 1:
            print('Depth service ready:', health)
            break
    except (OSError, ValueError):
        pass
    time.sleep(1)
else:
    depth_server_process.terminate()
    raise RuntimeError(f'Depth warmup timed out; inspect {DEPTH_SERVER_LOG}')

Loading and warming FastFS once. First compilation may take several minutes.
Depth service ready: {'model': {'checkpoint': 'model_best_bp2_serialize.pth', 'checkpoint_sha256': 'af0658f289ec840b292645f8d5538978f06e8cabaa1fd31e84acc91af268e990', 'max_disp': 192, 'units': 'metres', 'valid_iters': 8}, 'protocol_version': 1, 'status': 'ready'}


This exact request/response shape (`left_png_b64`/`right_png_b64`/
`intrinsics_flat`/`baseline_m` request, `depth_npy_b64` response) is what
`depth_camera.DepthCameraClient.compute_depth()` (Plan 12-06 Task 1) sends
and parses -- keep them in sync if either side changes.

### Step 9 — Colab: open a second tunnel

In [15]:
from pyngrok import ngrok

# A DIFFERENT local port than Step 3's gRPC server (5173), and a plain HTTP
# tunnel type -- unlike Step 4's gRPC tunnel, which specifically needs "tcp"
# for HTTP/2 semantics, this is a plain JSON-over-HTTP POST endpoint, so
# ngrok's default HTTP tunnel type works fine here.
depth_tunnel = ngrok.connect(3000, "http")
print(f"Depth endpoint reachable at: {depth_tunnel.public_url}/depth")

Depth endpoint reachable at: https://runway-uninsured-blank.ngrok-free.dev/depth


Pass the printed `.../depth` URL as `depth_camera.py`'s `--endpoint` argument
(Plan 12-06 Task 1) or `run_vla_episode.py`'s `--depth-endpoint` flag
(Task 3).

## Run the local episode with depth enabled

Complete Steps 1–5 and 7–9 first. Do not run teardown until the local process exits.
Use the updated local checkout containing the async recorder; uploading only this
notebook does not update the laptop code. In the local `control/` directory:

```bash
.venv/bin/python run_vla_episode.py --out outputs/depth_episode_UNIQUE_NAME \
  --server-address POLICY_TCP_HOST:PORT \
  --checkpoint victorvanhalst/smolvla_so101_cube \
  --stereo-camera-index "CCB Camera" \
  --depth-endpoint https://DEPTH_TUNNEL/depth \
  --depth-calibration stereo_calibration.json --depth-every-n-steps 1 \
  --max-steps 60 --instruction "Pick the red cube and place it in the bowl"
```

Use the actual addresses printed by the tunnel cells and a fresh output directory.
The depth health check runs before hardware opens. Each eligible step submits depth
unless a prior request is outstanding; busy skips are explicit, never a backlog.
Motion continues during requests. At shutdown the arm returns to start and the last
request is drained before writing the final depth summary.

Inspect `depth_summary.json`: `complete` must be positive; review `failed` and
`skipped_busy`. `episode.jsonl` links each submitted request to
`depth_overhead/STEP.json`, which records the final success/failure and depth file.
`depth.jsonl` indexes these completed results. `camera_depth_aligned_left/` and
`camera_depth_aligned_right/` are the exact rectified/resized depth inputs;
`depth_calibration.json` preserves calibration. Depth is float32 metres with NaN
for invalid pixels. `action_details.waypoint_sends` records send outcomes;
`raw_model_output` retains the policy output before validation. Driver sends are
not proof of measured physical arrival. This fixes recording; it does not add a
depth input feature to the existing RGB-only SmolVLA checkpoint.

A successful test requires saved valid depth throughout the episode AND observed
smooth motion. Check the manifest timings and the physical run; a normal step-count
termination by itself does not establish success.


### Teardown (extends Step 6 above)

Every session, no exceptions, ALSO tear down this second tunnel and stop the
Flask server process -- same rationale as the existing gRPC tunnel's own
teardown discipline (an unattended open tunnel into a Colab-hosted endpoint
is an elevation-of-privilege risk):

In [ ]:
# Run ONLY when the episode has finished and local depth results have drained.
if 'depth_server_process' in globals() and depth_server_process.poll() is None:
    depth_server_process.terminate()
    try:
        depth_server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        depth_server_process.kill()
        depth_server_process.wait()
if 'depth_tunnel' in globals():
    ngrok.disconnect(depth_tunnel.public_url)

Then interrupt/stop the Flask `app.run(...)` cell (Colab: click the cell's
stop button, or restart the runtime) so the depth server process itself is
not left listening after the tunnel is torn down.

---

## Summary

| Item             | Value                                                                                                                            |
| ---------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| Checkpoint       | `victorvanhalst/smolvla_so101_cube`                                                                                            |
| Task instruction | "Pick the red cube and place it in the bowl"                                                                                     |
| Tunnel type      | TCP (`ngrok.connect(5173, "tcp")`) — not HTTP                                                                                 |
| Camera mapping   | `camera1`=wrist, `camera2`=AR0144 stereo-left, `camera3`=AR0144 stereo-right                                               |
| Fallback tunnel  | `cloudflared` + configured Zero Trust tunnel (heavier setup)                                                                   |
| Fallback policy  | ACT or pi0/pi05 (`lerobot.policies`, already installed) if this checkpoint underperforms                                       |
| Depth endpoint   | Fast-FoundationStereo (NVlabs/Fast-FoundationStereo), served via Flask, HTTP tunnel (separate from the gRPC PolicyServer tunnel) |

In [17]:
!tail -100 /content/policy_server.log

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-09-30 05:48:51 y_server.py:420 {'fps': 30,
 'host': '0.0.0.0',
 'inference_latency': 0.03333333333333333,
 'obs_queue_timeout': 2,
 'port': 5173}
INFO 2026-09-30 05:48:51 y_server.py:430 PolicyServer started on 0.0.0.0:5173
INFO 2026-09-30 05:55:48 y_server.py:110 Client ipv6:%5B::1%5D:57210 connected and ready
INFO 2026-09-30 05:55:48 y_server.py:136 Receiving policy instructions from ipv6:%5B::1%5D:57210 | Policy type: smolvla | Pretrained name or path: victorvanhalst/smolvla_so101_cube | Actions per chunk: 50 | Device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary

In [20]:
!pip show torch nvidia-cudnn-cu12 | grep -E "^(Name|Version)"
!python -c "import torch; print('torch', torch.__version__); print('cudnn', torch.backends.cudnn.version())"

Name: torch
Version: 2.11.0+cu128
Name: nvidia-cudnn-cu12
Version: 9.19.0.56
torch 2.11.0+cu128
cudnn 91900


In [19]:
!{FASTFS_PYTHON} -c "import torch; print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

cuda available: True
device: NVIDIA L4


In [21]:
policy_server_process.poll()

In [22]:
!tail -40 /content/policy_server.log

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-09-29 10:51:52 y_server.py:420 {'fps': 30,
 'host': '0.0.0.0',
 'inference_latency': 0.03333333333333333,
 'obs_queue_timeout': 2,
 'port': 5173}
INFO 2026-09-29 10:51:52 y_server.py:430 PolicyServer started on 0.0.0.0:5173
INFO 2026-09-29 11:02:56 y_server.py:110 Client ipv6:%5B::1%5D:35870 connected and ready
INFO 2026-09-29 11:02:56 y_server.py:136 Receiving policy instructions from ipv6:%5B::1%5D:35870 | Policy type: smolvla | Pretrained name or path: victorvanhalst/smolvla_so101_cube | Actions per chunk: 50 | Device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary

## Step 6 — Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted `PolicyServer` is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day:

In [26]:
# Colab notebook cell, at the end of the session
ngrok.disconnect(tunnel.public_url)
ngrok.kill()

# PolicyServer now runs as a real subprocess (Step 3 fix, 2026-09-27), not a
# background thread -- terminate it explicitly so it doesn't keep holding
# GPU memory after the tunnel is torn down.
if "policy_server_process" in dir() and policy_server_process.poll() is None:
    policy_server_process.terminate()
    try:
        policy_server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        policy_server_process.kill()
    print("PolicyServer subprocess terminated.")